# Redução de Dimensionalidade, Seleção de Atributos, Classificação e Agrupamento

**Computational Learning and Pattern Recognition — UNESP**

Este notebook aborda:
1. **Redução de dimensionalidade** via PCA e **seleção de atributos** via SFFS
2. **Classificação** com 4 métodos: PolyMap-RBFNet-SVMLin, Random Neurons (ELM), SVM-RBF, KNN
3. **Agrupamento** com 4 métodos: BSAS, Fuzzy K-Means, K-Means, Ward Hierárquico

**Conjuntos de dados:**
- `make_classification` com **30 dimensões** (alta dimensionalidade artificial)
- **Olivetti Faces** (400 amostras, 4096 features, 40 classes)

**Métodos obrigatórios:**
- Classificação: `PolyMap-RBFNet-SVMLin` (Atividade III) e `Random Neurons` (Atividade IV)
- Agrupamento: `BSAS` e `Fuzzy K-Means` (Atividade V)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification, fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix,
                              silhouette_score, calinski_harabasz_score, v_measure_score)

np.random.seed(42)

## Seção 1 — Conjuntos de Dados

### 1.1 Conjunto Artificial (`make_classification`, 30 dimensões)

**Configuração e justificativa:**
- `n_samples=600`: volume suficiente para divisão treino/teste com representatividade por classe
- `n_features=30`: alta dimensionalidade explícita — objetivo central do experimento
- `n_informative=10`: apenas 1/3 das features carregam informação discriminativa real
- `n_redundant=10`: combinações lineares das informativas (simulam colinearidade)
- `n_repeated=5`: cópias exatas de features informativas (máxima redundância)
- `n_classes=4`: problema multiclasse moderado (15 features irrelevantes/redundantes/repetidas)

A escolha simula um cenário realista onde apenas 33% dos atributos são úteis, testando a capacidade do PCA e SFFS de identificar e eliminar o ruído dimensional.

In [ ]:
# --- Conjunto Artificial ---
N_SAMPLES   = 600
N_FEATURES  = 30
N_INFORM    = 10   # informativos
N_REDUND    = 10   # redundantes (combinações lineares)
N_REPEATED  = 5    # repetidos (cópias exatas)
N_CLASSES   = 4
# Irrelevantes = 30 - 10 - 10 - 5 = 5

X_art, y_art = make_classification(
    n_samples=N_SAMPLES, n_features=N_FEATURES,
    n_informative=N_INFORM, n_redundant=N_REDUND,
    n_repeated=N_REPEATED, n_classes=N_CLASSES,
    n_clusters_per_class=1, random_state=42
)

scaler_art = StandardScaler()
X_art = scaler_art.fit_transform(X_art)

X_art_D, X_art_I, y_art_D, y_art_I = train_test_split(
    X_art, y_art, test_size=0.33, stratify=y_art, random_state=42
)
print(f"Artificial | Total: {X_art.shape} | Treino: {X_art_D.shape} | Teste: {X_art_I.shape}")
print(f"  Composição: {N_INFORM} informativos, {N_REDUND} redundantes, {N_REPEATED} repetidos, 5 irrelevantes")

In [ ]:
# --- Olivetti Faces ---
olivetti = fetch_olivetti_faces(shuffle=True, random_state=42)
X_oliv_raw = olivetti.data    # 400 amostras, 4096 features (64x64 pixels)
y_oliv     = olivetti.target  # 40 classes (pessoas)

scaler_oliv = StandardScaler()
X_oliv = scaler_oliv.fit_transform(X_oliv_raw)

X_oliv_D, X_oliv_I, y_oliv_D, y_oliv_I = train_test_split(
    X_oliv, y_oliv, test_size=0.33, stratify=y_oliv, random_state=42
)
print(f"Olivetti | Total: {X_oliv.shape} | Treino: {X_oliv_D.shape} | Teste: {X_oliv_I.shape}")
print(f"  Classes: {len(np.unique(y_oliv))} pessoas")

In [ ]:
# --- Visualização dos datasets ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Artificial: distribuição PCA-2D
pca_viz = PCA(n_components=2, random_state=42)
X_art_2d = pca_viz.fit_transform(X_art)
cores = ['royalblue', 'tomato', 'green', 'orange']
for c in range(N_CLASSES):
    pos = y_art == c
    axes[0].scatter(X_art_2d[pos, 0], X_art_2d[pos, 1],
                    c=cores[c], label=f'ω{c+1}', alpha=0.6, s=20)
axes[0].set_title('Artificial (PCA-2D)', fontsize=13)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=8)

# Olivetti: algumas faces de exemplo
n_show = 8
for i in range(n_show):
    ax = fig.add_axes([0.36 + i * 0.04, 0.15, 0.038, 0.7])
    ax.imshow(olivetti.images[i], cmap='gray')
    ax.axis('off')
    if i == 0:
        ax.set_title('Olivetti Faces', fontsize=13, x=2.0, y=1.05)

# Distribuição de classes
axes[2].bar(range(N_CLASSES), [np.sum(y_art == c) for c in range(N_CLASSES)],
            color=cores, alpha=0.8)
axes[2].set_title('Distribuição de classes (Artificial)', fontsize=13)
axes[2].set_xlabel('Classe'); axes[2].set_ylabel('Amostras')
axes[2].set_xticks(range(N_CLASSES))
axes[2].set_xticklabels([f'ω{c+1}' for c in range(N_CLASSES)])

plt.tight_layout()
plt.savefig('fig_datasets.png', dpi=150, bbox_inches='tight')
plt.show()

## Seção 2 — Redução de Dimensionalidade

### 2.1 PCA — Principal Component Analysis

Retenção de 95% da variância explicada. O número de componentes é determinado automaticamente.

In [ ]:
def apply_pca(X_train, X_test, var_threshold=0.95):
    pca = PCA(n_components=var_threshold, random_state=42)
    X_tr = pca.fit_transform(X_train)
    X_te = pca.transform(X_test)
    return X_tr, X_te, pca

X_art_D_pca, X_art_I_pca, pca_art   = apply_pca(X_art_D, X_art_I)
X_oliv_D_pca, X_oliv_I_pca, pca_oliv = apply_pca(X_oliv_D, X_oliv_I)

print(f"PCA Artificial : {N_FEATURES} → {X_art_D_pca.shape[1]} componentes "
      f"({np.sum(pca_art.explained_variance_ratio_):.1%} da variância)")
print(f"PCA Olivetti   : {X_oliv_D.shape[1]} → {X_oliv_D_pca.shape[1]} componentes "
      f"({np.sum(pca_oliv.explained_variance_ratio_):.1%} da variância)")

In [ ]:
# Visualização: curvas de variância explicada acumulada
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, pca_full, title, n_orig in [
    (axes[0], PCA(random_state=42).fit(X_art_D),  'Artificial (30 features)', 30),
    (axes[1], PCA(random_state=42).fit(X_oliv_D), 'Olivetti Faces (4096 features)', 4096),
]:
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    ax.plot(range(1, len(cumvar) + 1), cumvar, 'b-', linewidth=1.5)
    ax.axhline(0.95, color='red', linestyle='--', label='95% limiar')
    idx_95 = np.searchsorted(cumvar, 0.95) + 1
    ax.axvline(idx_95, color='orange', linestyle=':', label=f'{idx_95} componentes')
    ax.fill_between(range(1, len(cumvar) + 1), cumvar, 0, alpha=0.1)
    ax.set_xlabel('Número de componentes', fontsize=12)
    ax.set_ylabel('Variância explicada acumulada', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('fig_pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 SFFS — Sequential Floating Forward Selection

**Critério:** $J = |\mathbf{V}_e + \mathbf{V}_i| \; / \; |\mathbf{V}_i|$ (razão de determinantes das scatter matrices)

**Diferença entre SFS e SFFS:** O SFS só avança (adiciona uma feature por vez). O SFFS adiciona uma feature e, em seguida, verifica se remover alguma das features já selecionadas melhora $J$ — permitindo corrigir escolhas subótimas anteriores (passo "flutuante" de retrocesso).

**Nota sobre o Olivetti Faces:** Com 4096 features originais, o SFFS seria computacionalmente inviável. Aplicamos SFFS sobre os componentes PCA (espaço reduzido), o que é uma prática comum em pipelines de alta dimensionalidade.

In [ ]:
def compute_scatter_J(feat_idx, X, pos_per_class):
    """J = |Ve + Vi| / |Vi| com scatter matrices."""
    if len(feat_idx) == 0:
        return 0.0
    idx = list(feat_idx)
    d = len(idx)
    m = X.shape[0]
    mu = np.mean(X[:, idx], axis=0, keepdims=True)  # (1, d)
    Sb = np.zeros((d, d))
    Sw = np.zeros((d, d))
    for pos in pos_per_class:
        _X = X[pos][:, idx]
        ni = len(pos)
        mu_i = np.mean(_X, axis=0, keepdims=True)
        diff = mu - mu_i
        Sb += (ni / m) * diff.T @ diff
        if ni > 1:
            Sw += (ni / m) * np.cov(_X.T).reshape(d, d)
    try:
        det_w = np.linalg.det(Sw)
        if abs(det_w) < 1e-12:
            det_w = 1e-12
        return np.linalg.det(Sb + Sw) / det_w
    except Exception:
        return 0.0


def sffs(X_train, y_train, n_select, candidate_attrs=None, verbose=False):
    """Sequential Floating Forward Selection."""
    if candidate_attrs is None:
        candidate_attrs = list(range(X_train.shape[1]))
    classes = np.unique(y_train)
    pos_per_class = [np.where(y_train == c)[0] for c in classes]

    Z = []   # selecionados
    W = list(candidate_attrs)  # candidatos restantes

    for step in range(min(n_select, len(W))):
        if not W:
            break

        # --- Passo forward: adiciona a feature que maximiza J ---
        best_j, best_feat = -np.inf, None
        for feat in W:
            j = compute_scatter_J(Z + [feat], X_train, pos_per_class)
            if j > best_j:
                best_j, best_feat = j, feat
        if best_feat is None:
            break
        Z.append(best_feat)
        W.remove(best_feat)
        if verbose:
            print(f"  [+] Adicionada feature {best_feat} | |Z|={len(Z)} | J={best_j:.4f}")

        # --- Passo backward flutuante: remove feature se melhora J ---
        while len(Z) > 1:
            j_cur = compute_scatter_J(Z, X_train, pos_per_class)
            best_rem_j, feat_rem = -np.inf, None
            for feat in Z[:-1]:  # não remove a última adicionada
                j_wo = compute_scatter_J([f for f in Z if f != feat], X_train, pos_per_class)
                if j_wo > best_rem_j:
                    best_rem_j, feat_rem = j_wo, feat
            if feat_rem is not None and best_rem_j > j_cur:
                Z.remove(feat_rem)
                W.append(feat_rem)
                if verbose:
                    print(f"  [-] Removida feature {feat_rem} | |Z|={len(Z)} | J={best_rem_j:.4f}")
            else:
                break

    return sorted(Z)

In [ ]:
# SFFS no conjunto artificial (30 features → 10 selecionadas)
N_SEL_ART = 10
print(f"Executando SFFS no conjunto artificial ({N_FEATURES} features → {N_SEL_ART} selecionadas)...")
sel_art = sffs(X_art_D, y_art_D, n_select=N_SEL_ART, verbose=True)
X_art_D_sffs = X_art_D[:, sel_art]
X_art_I_sffs = X_art_I[:, sel_art]
print(f"\nSFFS Artificial: features selecionadas = {sel_art}")

In [ ]:
# SFFS no Olivetti: aplicado sobre componentes PCA (inviável sobre 4096 features originais)
N_SEL_OLIV = 20
print(f"Executando SFFS no Olivetti (sobre {X_oliv_D_pca.shape[1]} componentes PCA → {N_SEL_OLIV} selecionados)...")
sel_oliv = sffs(X_oliv_D_pca, y_oliv_D, n_select=N_SEL_OLIV, verbose=False)
X_oliv_D_sffs = X_oliv_D_pca[:, sel_oliv]
X_oliv_I_sffs = X_oliv_I_pca[:, sel_oliv]
print(f"SFFS Olivetti: componentes PCA selecionados = {sel_oliv}")

In [ ]:
# Resumo das dimensionalidades
print("\n=== Resumo das dimensionalidades ===")
print(f"{'Dataset':<20} {'Original':>10} {'PCA':>10} {'SFFS':>10}")
print("-" * 52)
print(f"{'Artificial':<20} {X_art_D.shape[1]:>10} {X_art_D_pca.shape[1]:>10} {X_art_D_sffs.shape[1]:>10}")
print(f"{'Olivetti Faces':<20} {X_oliv_D.shape[1]:>10} {X_oliv_D_pca.shape[1]:>10} {X_oliv_D_sffs.shape[1]:>10}")

## Seção 3 — Classificação

### Métodos implementados

| Método | Descrição | Atividade |
|--------|-----------|----------|
| **PolyMap-RBFNet-SVMLin** | Mapeamento polinomial → RBF (centróides K-Means) → saída SSE (≡ SVMLin) | III |
| **Random Neurons (ELM)** | RBF com centróides aleatórios → saída SSE (Extreme Learning Machine) | IV |
| **SVM-RBF** | SVM com kernel RBF (scikit-learn) | III |
| **KNN** | K-Vizinhos Mais Próximos (k=5) | livre |

**PolyMap-RBFNet-SVMLin:** Pipeline que combina mapeamento polinomial (PolyMap), rede com funções de base radial inicializadas via K-Means (RBFNet) e treinamento da camada de saída por mínimos quadrados, equivalente ao SVM linear no espaço mapeado (SVMLin).  
**Random Neurons (ELM):** Variante da RBFNet onde os centróides são *aleatoriamente* amostrados do conjunto de treino (sem K-Means). Apenas os pesos da camada de saída são treinados, via SSE.

In [ ]:
# ========== Funções auxiliares: RBF hidden layer e treinamento SSE ==========

def kmeans_centers(X, k, max_iter=300, tol=1e-4):
    """K-Means simplificado para inicialização dos centróides."""
    idx = np.random.choice(len(X), k, replace=False)
    centroids = X[idx].astype(float).copy()
    for _ in range(max_iter):
        dists = np.array([[np.linalg.norm(x - c) for c in centroids] for x in X])
        labels = dists.argmin(axis=1)
        new_c = np.array([
            X[labels == j].mean(axis=0) if (labels == j).any() else centroids[j]
            for j in range(k)
        ])
        if np.linalg.norm(new_c - centroids) < tol:
            break
        centroids = new_c
    # Raio = distância média dos pontos ao centróide
    sigmas = np.array([
        max(np.mean([np.linalg.norm(X[i] - centroids[j])
                     for i in np.where(labels == j)[0]]), 1e-4)
        if (labels == j).any() else 1.0
        for j in range(k)
    ])
    return centroids, sigmas


def rbf_layer(X, centroids, sigmas):
    """Saída da camada oculta RBF (Gaussiana)."""
    m, k = len(X), len(centroids)
    G = np.zeros((m, k))
    for j in range(k):
        d2 = np.sum((X - centroids[j]) ** 2, axis=1)
        G[:, j] = np.exp(-d2 / (2 * sigmas[j] ** 2))
    return G


def train_output_sse(G, y):
    """Treina pesos da saída via SSE: W = (GᵀG)⁻¹GᵀY."""
    classes = np.unique(y)
    Y = np.zeros((len(y), len(classes)))
    for i, c in enumerate(classes):
        Y[y == c, i] = 1
    W, _, _, _ = np.linalg.lstsq(G, Y, rcond=None)
    return W, classes


def predict_rbf(X, centroids, sigmas, W, classes):
    G = rbf_layer(X, centroids, sigmas)
    scores = G @ W
    return classes[scores.argmax(axis=1)]

In [ ]:
# ========== Classificador 1: PolyMap-RBFNet-SVMLin ==========
class PolyMapRBFNetSVMLin:
    """
    Pipeline: mapeamento polinomial (grau 2) → camada oculta RBF (centróides K-Means)
    → saída treinada por SSE (equivalente a SVM Linear no espaço mapeado).
    O PolyMap expande a representação antes da RBFNet, aumentando a capacidade
    de separação via Teorema de Cover.
    """
    def __init__(self, n_hidden=25, poly_degree=2, max_poly_features=200):
        self.n_hidden = n_hidden
        self.poly_degree = poly_degree
        self.max_poly_features = max_poly_features

    def fit(self, X, y):
        # Mapeamento polinomial (PolyMap)
        self.poly_ = PolynomialFeatures(degree=self.poly_degree, include_bias=False)
        Xp = self.poly_.fit_transform(X)
        # Trunca se necessário (evita explosão dimensional)
        self.n_poly_used_ = min(Xp.shape[1], self.max_poly_features)
        Xp = Xp[:, :self.n_poly_used_]
        # RBFNet: K-Means para centróides
        self.centroids_, self.sigmas_ = kmeans_centers(Xp, self.n_hidden)
        G = rbf_layer(Xp, self.centroids_, self.sigmas_)
        # SVMLin: treinamento SSE da saída
        self.W_, self.classes_ = train_output_sse(G, y)
        return self

    def predict(self, X):
        Xp = self.poly_.transform(X)[:, :self.n_poly_used_]
        return predict_rbf(Xp, self.centroids_, self.sigmas_, self.W_, self.classes_)


# ========== Classificador 2: Random Neurons (ELM) ==========
class RandomNeuronsELM:
    """
    Extreme Learning Machine: centróides RBF ALEATORIAMENTE amostrados do conjunto
    de treinamento (sem K-Means). Apenas os pesos da camada de saída são treinados.
    Diferença fundamental em relação à RBFNet: não há fase não-supervisionada.
    """
    def __init__(self, n_hidden=80):
        self.n_hidden = n_hidden

    def fit(self, X, y):
        np.random.seed(42)
        idx = np.random.choice(len(X), self.n_hidden, replace=True)
        self.centroids_ = X[idx].astype(float).copy()
        # Sigma global: desvio padrão dos dados escalado pelo número de neurônios
        self.sigma_ = max(np.std(X) * np.sqrt(X.shape[1] / self.n_hidden), 1e-4)
        self.sigmas_ = np.full(self.n_hidden, self.sigma_)
        G = rbf_layer(X, self.centroids_, self.sigmas_)
        self.W_, self.classes_ = train_output_sse(G, y)
        return self

    def predict(self, X):
        return predict_rbf(X, self.centroids_, self.sigmas_, self.W_, self.classes_)

In [ ]:
# ========== Função de avaliação ==========
def evaluate_clf(clf, X_tr, y_tr, X_te, y_te):
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    return accuracy_score(y_te, y_pred), cohen_kappa_score(y_te, y_pred), y_pred


def run_classification(X_orig_D, y_D, X_orig_I, y_I,
                       X_pca_D, X_pca_I,
                       X_sffs_D, X_sffs_I,
                       n_hidden=25):
    """Executa os 4 classificadores nos 3 conjuntos de dados."""
    classifiers = [
        ('PolyMap-RBFNet-SVMLin',  PolyMapRBFNetSVMLin(n_hidden=n_hidden)),
        ('Random Neurons (ELM)',   RandomNeuronsELM(n_hidden=n_hidden * 3)),
        ('SVM-RBF',                SVC(kernel='rbf', C=10, gamma='scale', random_state=42)),
        ('KNN (k=5)',              KNeighborsClassifier(n_neighbors=5)),
    ]
    data_versions = [
        ('Original', X_orig_D, X_orig_I),
        ('PCA',      X_pca_D,  X_pca_I),
        ('SFFS',     X_sffs_D, X_sffs_I),
    ]
    rows = []
    preds = {}
    for clf_name, clf in classifiers:
        for data_name, X_tr, X_te in data_versions:
            oa, kappa, y_pred = evaluate_clf(clf, X_tr, y_D, X_te, y_I)
            rows.append({'Método': clf_name, 'Dados': data_name,
                         'OA': oa, 'Kappa': kappa})
            preds[(clf_name, data_name)] = y_pred
            print(f"  {clf_name:<30} | {data_name:<10} | OA={oa:.4f} | κ={kappa:.4f}")
    return pd.DataFrame(rows), preds

In [ ]:
print("\n=== CLASSIFICAÇÃO — Conjunto Artificial ===")
df_clf_art, preds_art = run_classification(
    X_art_D, y_art_D, X_art_I, y_art_I,
    X_art_D_pca, X_art_I_pca,
    X_art_D_sffs, X_art_I_sffs,
    n_hidden=25
)

In [ ]:
print("\n=== CLASSIFICAÇÃO — Olivetti Faces ===")
df_clf_oliv, preds_oliv = run_classification(
    X_oliv_D, y_oliv_D, X_oliv_I, y_oliv_I,
    X_oliv_D_pca, X_oliv_I_pca,
    X_oliv_D_sffs, X_oliv_I_sffs,
    n_hidden=40
)

In [ ]:
# Tabela resumo formatada
def pivot_results(df, metric='OA'):
    tbl = df.pivot(index='Método', columns='Dados', values=metric)
    tbl = tbl[['Original', 'PCA', 'SFFS']]
    return tbl.round(4)

print("\n--- Acurácia Global (OA) — Artificial ---")
print(pivot_results(df_clf_art, 'OA').to_string())
print("\n--- Coef. Kappa — Artificial ---")
print(pivot_results(df_clf_art, 'Kappa').to_string())

print("\n--- Acurácia Global (OA) — Olivetti ---")
print(pivot_results(df_clf_oliv, 'OA').to_string())
print("\n--- Coef. Kappa — Olivetti ---")
print(pivot_results(df_clf_oliv, 'Kappa').to_string())

In [ ]:
# Visualização: gráfico de barras comparativo
def plot_clf_results(df_art, df_oliv, metric='OA', ylabel='Acurácia Global (OA)'):
    methods   = df_art['Método'].unique()
    data_vers = ['Original', 'PCA', 'SFFS']
    x = np.arange(len(methods))
    width = 0.25
    colors = ['steelblue', 'tomato', 'seagreen']

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for ax, df, title in [(axes[0], df_art, 'Artificial'), (axes[1], df_oliv, 'Olivetti Faces')]:
        tbl = df.pivot(index='Método', columns='Dados', values=metric).reindex(methods)
        for i, (dv, col) in enumerate(zip(data_vers, colors)):
            ax.bar(x + i * width, tbl[dv], width, label=dv, color=col, alpha=0.85)
        ax.set_xticks(x + width)
        ax.set_xticklabels([m.replace(' (', '\n(') for m in methods], fontsize=9)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(f'{title}', fontsize=13)
        ax.legend(fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3, axis='y')
        # Adiciona valores sobre as barras
        for i, dv in enumerate(data_vers):
            for j, m in enumerate(methods):
                val = tbl.loc[m, dv]
                if not np.isnan(val):
                    ax.text(j + i * width, val + 0.01, f'{val:.2f}',
                            ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(f'fig_clf_{metric.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_clf_results(df_clf_art, df_clf_oliv, 'OA', 'Acurácia Global (OA)')
plot_clf_results(df_clf_art, df_clf_oliv, 'Kappa', 'Coeficiente Kappa (κ)')

In [ ]:
# Matriz de confusão: melhor configuração em cada dataset
def plot_best_confusion(df, preds, y_true, dataset_name):
    # Identifica a melhor combinação (método + dados) por OA
    best_row = df.loc[df['OA'].idxmax()]
    best_key = (best_row['Método'], best_row['Dados'])
    y_pred = preds[best_key]
    labels = np.unique(y_true)
    cm_mat = confusion_matrix(y_true, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_mat, cmap='Blues', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=7 if len(labels) > 10 else 10)
    ax.set_yticklabels(labels, fontsize=7 if len(labels) > 10 else 10)
    ax.set_xlabel('Classe Predita', fontsize=12)
    ax.set_ylabel('Classe Real', fontsize=12)
    ax.set_title(f'{dataset_name} — {best_key[0]}\n({best_key[1]}) OA={best_row["OA"]:.4f}', fontsize=11)
    if len(labels) <= 10:
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, cm_mat[i, j], ha='center', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'fig_confmat_{dataset_name.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_best_confusion(df_clf_art, preds_art, y_art_I, 'Artificial')
plot_best_confusion(df_clf_oliv, preds_oliv, y_oliv_I, 'Olivetti')

### Discussão dos resultados de classificação

**Impacto da alta dimensionalidade:**  
No conjunto artificial com 30 features (apenas 10 realmente informativas), espera-se que métodos baseados em distância (KNN) sofram mais com a maldição da dimensionalidade — as distâncias se tornam menos discriminativas em espaços de alta dimensão.  

**Influência do PCA:**  
O PCA elimina as dimensões de menor variância, reduzindo ruído e correlações. Como o conjunto artificial possui features redundantes e repetidas, o PCA deve melhorar ou manter o desempenho com expressiva redução dimensional.  

**Influência do SFFS:**  
O SFFS maximiza o critério discriminativo $J$ entre classes, selecionando diretamente as features mais relevantes para separação. Para o conjunto artificial, espera-se que o SFFS identifique subconjunto próximo das 10 features informativas originais.  

**PolyMap-RBFNet-SVMLin vs Random Neurons (ELM):**  
A principal diferença entre os dois está na inicialização dos centróides RBF: K-Means (estruturado) vs. aleatório. A inicialização K-Means adapta os centróides à distribuição dos dados, enquanto o ELM compensa com mais neurônios e funciona como um estimador não-paramétrico aleatório.

## Seção 4 — Agrupamento

### Métodos implementados

| Método | Descrição | Atividade |
|--------|-----------|----------|
| **BSAS** | Algoritmo sequencial básico com limiar τ | V |
| **Fuzzy K-Means (FKM)** | K-Means com pertinências contínuas (β=2) | V |
| **K-Means** | Particionamento por centróides (sklearn) | livre |
| **Ward Hierárquico** | Aglomerativo minimizando variância intra-cluster | livre |

**Métricas de avaliação:**
- **Silhouette Coefficient** $\in [-1, 1]$: coesão vs. separação dos clusters (maior = melhor, sem referência)
- **Calinski-Harabász Index**: razão entre dispersão inter/intra-cluster (maior = melhor, sem referência)
- **V-measure** $\in [0, 1]$: combinação de homogeneidade e completude (requer rótulos verdadeiros)

In [ ]:
# ========== BSAS ==========
def bsas(X, tau, max_clusters):
    """Basic Sequential Algorithmic Scheme."""
    n = len(X)
    labels    = -np.ones(n, dtype=int)
    centroids = [X[0].copy()]
    counts    = [1]
    labels[0] = 0

    for i in range(1, n):
        dists = [np.linalg.norm(X[i] - c) for c in centroids]
        k     = int(np.argmin(dists))
        if dists[k] > tau and len(centroids) < max_clusters:
            centroids.append(X[i].copy())
            counts.append(1)
            labels[i] = len(centroids) - 1
        else:
            labels[i] = k
            counts[k] += 1
            centroids[k] = centroids[k] + (X[i] - centroids[k]) / counts[k]
    return labels


# ========== Fuzzy K-Means ==========
def fuzzy_kmeans(X, k, beta=2.0, epsilon=1e-4, max_iter=300):
    """Fuzzy K-Means com fator de fuzzificação beta."""
    np.random.seed(42)
    m, _ = X.shape
    expo = 2.0 / (beta - 1.0)
    # Inicialização dos centróides
    idx = np.random.choice(m, k, replace=False)
    mu  = X[idx].astype(float).copy()

    for _ in range(max_iter):
        # Cálculo das dissimilaridades
        dists = np.array([[np.linalg.norm(X[i] - mu[j]) + 1e-9
                           for j in range(k)] for i in range(m)])  # (m, k)
        # Cálculo das pertinências λ
        lamb = np.zeros((m, k))
        for i in range(m):
            for j in range(k):
                lamb[i, j] = 1.0 / np.sum((dists[i, j] / dists[i]) ** expo)
        # Atualização dos centróides
        mu_new = np.zeros_like(mu)
        for j in range(k):
            lj = lamb[:, j] ** beta
            mu_new[j] = (lj @ X) / lj.sum()
        if np.linalg.norm(mu_new - mu) < epsilon:
            break
        mu = mu_new
    return lamb.argmax(axis=1)


# ========== Avaliação de clustering ==========
def eval_clustering(X, labels, y_true):
    n_clus = len(np.unique(labels))
    if n_clus < 2 or n_clus >= len(X):
        return {'N Clusters': n_clus, 'Silhouette': np.nan,
                'CH Index': np.nan, 'V-measure': np.nan}
    try:
        sil = silhouette_score(X, labels, sample_size=min(500, len(X)), random_state=42)
    except:
        sil = np.nan
    try:
        ch = calinski_harabasz_score(X, labels)
    except:
        ch = np.nan
    vm = v_measure_score(y_true, labels)
    return {'N Clusters': n_clus, 'Silhouette': sil, 'CH Index': ch, 'V-measure': vm}

In [ ]:
def run_clustering(X_orig, y_true, X_pca, X_sffs,
                   n_clusters, tau_factor=2.0, dataset_name=''):
    """
    Executa 4 métodos de agrupamento nos 3 conjuntos de dados.
    O tau do BSAS é estimado como tau_factor * std(X).
    """
    data_versions = [('Original', X_orig), ('PCA', X_pca), ('SFFS', X_sffs)]
    rows = []

    for data_name, X in data_versions:
        tau = tau_factor * np.std(X)
        print(f"\n  [{dataset_name} | {data_name}] shape={X.shape}, tau={tau:.4f}")

        # 1. BSAS
        lab = bsas(X, tau=tau, max_clusters=n_clusters * 2)
        r = eval_clustering(X, lab, y_true)
        r.update({'Método': 'BSAS', 'Dados': data_name}); rows.append(r)
        print(f"    BSAS          → {r['N Clusters']} clusters | Sil={r['Silhouette']:.4f} | V={r['V-measure']:.4f}")

        # 2. Fuzzy K-Means
        lab = fuzzy_kmeans(X, k=n_clusters)
        r = eval_clustering(X, lab, y_true)
        r.update({'Método': 'Fuzzy K-Means', 'Dados': data_name}); rows.append(r)
        print(f"    FKM           → {r['N Clusters']} clusters | Sil={r['Silhouette']:.4f} | V={r['V-measure']:.4f}")

        # 3. K-Means
        lab = KMeans(n_clusters=n_clusters, n_init=10, max_iter=500, random_state=42).fit_predict(X)
        r = eval_clustering(X, lab, y_true)
        r.update({'Método': 'K-Means', 'Dados': data_name}); rows.append(r)
        print(f"    K-Means       → {r['N Clusters']} clusters | Sil={r['Silhouette']:.4f} | V={r['V-measure']:.4f}")

        # 4. Ward Hierárquico
        lab = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward').fit_predict(X)
        r = eval_clustering(X, lab, y_true)
        r.update({'Método': 'Ward Hierárquico', 'Dados': data_name}); rows.append(r)
        print(f"    Ward          → {r['N Clusters']} clusters | Sil={r['Silhouette']:.4f} | V={r['V-measure']:.4f}")

    return pd.DataFrame(rows)

In [ ]:
print("\n=== AGRUPAMENTO — Conjunto Artificial (k=4) ===")
df_clust_art = run_clustering(
    X_art, y_art,
    np.vstack([X_art_D_pca, X_art_I_pca]),
    np.vstack([X_art_D_sffs, X_art_I_sffs]),
    n_clusters=N_CLASSES, tau_factor=2.0,
    dataset_name='Artificial'
)

In [ ]:
# Olivetti Faces: 40 classes é muito para FKM e BSAS (custo computacional)
# Usamos k=10 como compromisso; V-measure ainda avalia contra os 40 rótulos reais
N_CLUST_OLIV = 10
print(f"\n=== AGRUPAMENTO — Olivetti Faces (k={N_CLUST_OLIV}, avaliado contra 40 classes) ===")
df_clust_oliv = run_clustering(
    X_oliv, y_oliv,
    np.vstack([X_oliv_D_pca, X_oliv_I_pca]),
    np.vstack([X_oliv_D_sffs, X_oliv_I_sffs]),
    n_clusters=N_CLUST_OLIV, tau_factor=3.0,
    dataset_name='Olivetti'
)

In [ ]:
# Tabelas de resultados
for metric in ['Silhouette', 'V-measure']:
    print(f"\n--- {metric} — Artificial ---")
    tbl = df_clust_art.pivot(index='Método', columns='Dados', values=metric)
    tbl = tbl[['Original', 'PCA', 'SFFS']].round(4)
    print(tbl.to_string())

    print(f"\n--- {metric} — Olivetti ---")
    tbl = df_clust_oliv.pivot(index='Método', columns='Dados', values=metric)
    tbl = tbl[['Original', 'PCA', 'SFFS']].round(4)
    print(tbl.to_string())

In [ ]:
# Visualização: gráficos de barras para clustering
def plot_clust_results(df_art, df_oliv):
    metrics   = ['Silhouette', 'CH Index', 'V-measure']
    ylabels   = ['Silhouette', 'CH Index', 'V-measure']
    methods   = ['BSAS', 'Fuzzy K-Means', 'K-Means', 'Ward Hierárquico']
    data_vers = ['Original', 'PCA', 'SFFS']
    colors    = ['steelblue', 'tomato', 'seagreen']
    x = np.arange(len(methods))
    width = 0.25

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for row, (df, ds_name) in enumerate([(df_art, 'Artificial'), (df_oliv, 'Olivetti')]):
        for col, (metric, ylabel) in enumerate(zip(metrics, ylabels)):
            ax = axes[row, col]
            tbl = df.pivot(index='Método', columns='Dados', values=metric).reindex(methods)
            for i, (dv, col_c) in enumerate(zip(data_vers, colors)):
                vals = tbl[dv].values.astype(float)
                # Normaliza CH Index para visualização
                if metric == 'CH Index':
                    max_v = np.nanmax(np.abs(vals))
                    plot_vals = vals / max_v if max_v > 0 else vals
                    ax.set_title(f'{ds_name} — {ylabel} (normalizado)', fontsize=10)
                else:
                    plot_vals = vals
                    ax.set_title(f'{ds_name} — {ylabel}', fontsize=10)
                ax.bar(x + i * width, np.nan_to_num(plot_vals), width,
                       label=dv, color=col_c, alpha=0.85)
            ax.set_xticks(x + width)
            ax.set_xticklabels([m.replace(' ', '\n') for m in methods], fontsize=8)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('fig_clustering.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_clust_results(df_clust_art, df_clust_oliv)

In [ ]:
# Visualização: scatter plot dos agrupamentos no espaço PCA-2D (conjunto artificial)
pca2d = PCA(n_components=2, random_state=42)
X_art_2d_all = pca2d.fit_transform(X_art)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
cmap = plt.cm.get_cmap('tab10')

configs = [
    ('Original', X_art),
    ('PCA',  np.vstack([X_art_D_pca, X_art_I_pca])),
    ('SFFS', np.vstack([X_art_D_sffs, X_art_I_sffs])),
]

# Linhas: BSAS e K-Means (representativos)
for col, (data_name, X_use) in enumerate(configs):
    tau = 2.0 * np.std(X_use)

    lab_bsas = bsas(X_use, tau=tau, max_clusters=N_CLASSES * 2)
    lab_km   = KMeans(n_clusters=N_CLASSES, n_init=10, random_state=42).fit_predict(X_use)

    for row, (lab, method) in enumerate([(lab_bsas, 'BSAS'), (lab_km, 'K-Means')]):
        ax = axes[row, col]
        n_cl = len(np.unique(lab))
        for c in np.unique(lab):
            pos = lab == c
            ax.scatter(X_art_2d_all[pos, 0], X_art_2d_all[pos, 1],
                       c=[cmap(c / max(n_cl - 1, 1))], s=15, alpha=0.7)
        ax.set_title(f'{method} | {data_name} ({n_cl} grupos)', fontsize=10)
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('Agrupamentos no espaço PCA-2D — Conjunto Artificial', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig_clust_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### Discussão dos resultados de agrupamento

**BSAS:** Algoritmo sequencial sensível à ordem dos dados e ao parâmetro τ. Pode criar número variável de clusters. Com dados de alta dimensão, τ calculado como fator × std(X) pode não refletir adequadamente as distâncias reais — a maldição da dimensionalidade comprime as distâncias entre pontos, tornando τ menos discriminativo.

**Fuzzy K-Means:** Atribui graus de pertinência contínuos, o que é vantajoso quando as fronteiras entre clusters são graduais. No espaço de alta dimensão, a inicialização aleatória dos centróides pode levar a mínimos locais ruins.

**K-Means e Ward:** Métodos mais robustos e amplamente utilizados. O Ward minimiza a variância intra-cluster aglomerativamente, tendendo a produzir clusters mais compactos e balanceados do que K-Means com inicialização aleatória.

**Impacto do PCA e SFFS no agrupamento:**  
- PCA reduz a dimensionalidade preservando a maior variância, eliminando ruído dimensional que dificulta o agrupamento
- SFFS seleciona features com maior poder discriminativo entre classes, o que pode não coincidir com as melhores features para agrupamento (critério de scatter usa rótulos de classe)
- Em agrupamento *sem referência*, PCA tende a ser mais adequado que SFFS, pois não depende de rótulos

## Seção 5 — Discussão Geral e Conclusões

### 5.1 Impacto da alta dimensionalidade

O conjunto artificial com 30 features (10 informativas, 15 redundantes/repetidas, 5 irrelevantes) demonstra diretamente a problemática da alta dimensionalidade:
- **Para classificação:** métodos baseados em distância como KNN sofrem porque as distâncias se tornam aproximadamente iguais em espaços de alta dimensão
- **Para agrupamento:** BSAS e K-Means dependem criticamente de boas métricas de distância — que se degradam com o aumento de dimensão
- **Para o Olivetti Faces (4096 features):** o cenário é ainda mais extremo — a maior parte do volume do espaço está concentrada longe do centro, e qualquer método não-parametrizado sofre

### 5.2 PCA vs SFFS

| Aspecto | PCA | SFFS |
|---------|-----|------|
| Tipo | Extração (cria novos atributos) | Seleção (mantém atributos originais) |
| Interpretabilidade | Baixa (combinações lineares) | Alta (features originais preservadas) |
| Supervisão | Não-supervisionado | Supervisionado (requer rótulos) |
| Custo computacional | $O(d^2 \cdot m)$ | $O(d^2 \cdot n_{sel}^2)$ — pode ser alto |
| Para classificação | Bom (preserva variância) | Melhor (otimiza separabilidade) |
| Para clustering | Bom (sem dependência de rótulos) | Menos adequado (usa rótulos) |

### 5.3 Vantagens e limitações dos métodos de classificação

- **PolyMap-RBFNet-SVMLin:** Alta capacidade de mapeamento não-linear (Teorema de Cover). Limitação: custo computacional do mapeamento polinomial cresce combinatorialmente com o grau
- **Random Neurons (ELM):** Extremamente rápido de treinar. Limitação: a aleatoriedade implica alta variância — resultados podem variar entre execuções
- **SVM-RBF:** Boa generalização via margem máxima, robusto com kernel RBF. Limitação: sensível a γ e C
- **KNN:** Simples e sem treinamento. Limitação: sofre severamente com alta dimensionalidade e é lento na predição ($O(m \cdot d)$ por padrão)

### 5.4 Vantagens e limitações dos métodos de agrupamento

- **BSAS:** Sem necessidade de especificar k a priori, apenas τ. Limitação: sensível à ordem dos dados
- **FKM:** Pertinências graduais são mais realistas em dados com sobreposição. Limitação: lento para k grande e custo $O(m \cdot k)$ por iteração
- **K-Means:** Rápido e amplamente usado. Limitação: assume clusters esféricos de tamanho similar
- **Ward Hierárquico:** Produz dendrograma (permite análise em múltiplas escalas). Limitação: custo $O(m^2)$ — inviável para datasets grandes